In [1]:
import pandas as pd
from src.utils.paths import load_paths

paths = load_paths()

vnat_feats = pd.read_parquet(paths.data_processed / "vnat" / "features_trainable.parquet")
iscx_feats = pd.read_parquet(paths.data_processed / "iscx" / "features.parquet")

iscx_feats = iscx_feats[iscx_feats["q_min_packets_ok"] == 1.0].copy()

df_all = pd.concat([vnat_feats, iscx_feats], ignore_index=True)
df_all["split"] = df_all["split"].replace({
    "iscx_train": "train",
    "iscx_val": "val",
    "iscx_test": "test"
})

print("Splits after merging:")
print(df_all["split"].value_counts())
print("\nLabels:")
print(df_all["label"].value_counts())

Splits after merging:
split
train    16138
val       1926
test      1844
Name: count, dtype: int64

Labels:
label
0    16591
1     3317
Name: count, dtype: int64


In [2]:
from src.models.xgb_train import train_xgboost
from src.utils.logging import setup_logger

logger = setup_logger(level="INFO")
xgb_yaml = paths.configs_dir / "xgb.yaml"

res = train_xgboost(paths=paths, xgb_yaml=xgb_yaml, df=df_all)

print("Saved model:", res.model_path)
print("Saved metrics:", res.metrics_path)
print("Saved preds:", res.preds_path)

print("\nTrain (Combined):", res.metrics["splits"]["train"])
print("\nVal (Combined):", res.metrics["splits"]["val"])
print("\nTest (Combined):", res.metrics["splits"]["test"])
print("\nFirewall policy:", res.metrics.get("firewall_policy", res.metrics.get("policy_thresholds", {})))

DEBUG mean/std (first 5):
[-2.5   -2.108 -1.309 -0.72  -0.473]
[2.877 2.76  0.681 0.005 0.003]
[0]	train-logloss:0.65228	train-auc:0.97425	train-aucpr:0.94423	val-logloss:0.65596	val-auc:0.92851	val-aucpr:0.93082
[100]	train-logloss:0.06037	train-auc:0.99905	train-aucpr:0.99536	val-logloss:0.16032	val-auc:0.97884	val-aucpr:0.97374
[200]	train-logloss:0.02649	train-auc:0.99984	train-aucpr:0.99916	val-logloss:0.13527	val-auc:0.98383	val-aucpr:0.98072
[300]	train-logloss:0.01505	train-auc:0.99997	train-aucpr:0.99982	val-logloss:0.14333	val-auc:0.98334	val-aucpr:0.98002
[400]	train-logloss:0.00915	train-auc:0.99999	train-aucpr:0.99996	val-logloss:0.15781	val-auc:0.98400	val-aucpr:0.97976
[437]	train-logloss:0.00785	train-auc:1.00000	train-aucpr:0.99997	val-logloss:0.16107	val-auc:0.98439	val-aucpr:0.98017
Calibrating XGBoost probabilities (internal step)...
Saved model: C:\Users\scoti\PycharmProjects\ai-vpn-firewall\artifacts\xgb\model.json
Saved metrics: C:\Users\scoti\PycharmProjects\ai-

In [3]:
import json
import numpy as np
import xgboost as xgb
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix

from src.pipeline.artifacts import default_feature_artifacts
from src.pipeline.feature_pipeline import FeaturePipeline
from src.eval.metrics import pick_threshold_for_fpr, confusion_at_threshold

model_path = paths.repo_root / "artifacts" / "xgb" / "model.json"
booster = xgb.Booster()
booster.load_model(str(model_path))

feature_art = default_feature_artifacts(paths.artifacts_dir / "features")
pipeline = FeaturePipeline.load(feature_art)

X_all = pipeline.transform(df_all)
feat_cols = pipeline.model_feature_names()

def is_iscx(row):
    pass

iscx_test_ids = set(pd.read_parquet(paths.data_processed / "iscx" / "features.parquet")
                    [lambda d: d["split"] == "iscx_test"]["capture_id"])

iscx_test_mask = df_all["capture_id"].isin(iscx_test_ids)
iscx_test_df = df_all[iscx_test_mask].copy()

print(f"Recovered ISCX Test set: {len(iscx_test_df)} rows")

X_iscx = X_all.loc[iscx_test_df.index, feat_cols].to_numpy(dtype=float)
y_iscx = iscx_test_df["label"].to_numpy(dtype=int)

p_iscx = booster.predict(xgb.DMatrix(X_iscx, feature_names=feat_cols))

print("ISCX TEST ROC:", roc_auc_score(y_iscx, p_iscx))
print("ISCX TEST PR :", average_precision_score(y_iscx, p_iscx))

metrics_path = paths.repo_root / "artifacts" / "xgb" / "metrics.json"
m = json.loads(metrics_path.read_text(encoding="utf-8"))

firewall_pol = m.get("firewall_policy", {})
thr = float(firewall_pol.get("threshold", 0.5))
pol_name = firewall_pol.get("chosen", "unknown")

print("Using firewall policy:", pol_name, "threshold:", thr)

yhat = (p_iscx >= thr).astype(int)
tn, fp, fn, tp = confusion_matrix(y_iscx, yhat).ravel()

prec = tp / (tp + fp + 1e-9)
rec  = tp / (tp + fn + 1e-9)
fpr  = fp / (fp + tn + 1e-9)

print("ISCX TEST @ firewall thr:", {"tn":tn,"fp":fp,"fn":fn,"tp":tp, "precision":prec, "recall":rec, "fpr":fpr})

Recovered ISCX Test set: 1634 rows
ISCX TEST ROC: 0.9556370542486337
ISCX TEST PR : 0.9025588891746157
Using firewall policy: fpr_0_1pct threshold: 0.9527888920328592
ISCX TEST @ firewall thr: {'tn': np.int64(1396), 'fp': np.int64(0), 'fn': np.int64(103), 'tp': np.int64(135), 'precision': np.float64(0.9999999999925927), 'recall': np.float64(0.5672268907539192), 'fpr': np.float64(0.0)}


In [4]:
vnat_test_ids = set(pd.read_parquet(paths.data_processed / "vnat" / "features_trainable.parquet")
                    [lambda d: d["split"] == "test"]["capture_id"])

vnat_test_mask = df_all["capture_id"].isin(vnat_test_ids)
vnat_test_df = df_all[vnat_test_mask].copy()

print(f"Recovered VNAT Test set: {len(vnat_test_df)} rows")

X_vnat_test = X_all.loc[vnat_test_df.index, feat_cols].to_numpy(dtype=float)
y_vnat_test = vnat_test_df["label"].to_numpy(dtype=int)
p_vnat_test = booster.predict(xgb.DMatrix(X_vnat_test, feature_names=feat_cols))

yhat = (p_vnat_test >= thr).astype(int)
tn, fp, fn, tp = confusion_matrix(y_vnat_test, yhat).ravel()

prec = tp / (tp + fp + 1e-9)
rec  = tp / (tp + fn + 1e-9)
fpr  = fp / (fp + tn + 1e-9)

print("VNAT TEST @ firewall thr:", {"tn":tn,"fp":fp,"fn":fn,"tp":tp,
                                   "precision":prec,"recall":rec,"fpr":fpr})

Recovered VNAT Test set: 210 rows
VNAT TEST @ firewall thr: {'tn': np.int64(197), 'fp': np.int64(0), 'fn': np.int64(4), 'tp': np.int64(9), 'precision': np.float64(0.9999999998888889), 'recall': np.float64(0.6923076922544379), 'fpr': np.float64(0.0)}


In [5]:
def per_capture_recall(df_split, probs, thr):
    tmp = df_split[["capture_id", "label"]].copy()
    tmp["p"] = np.asarray(probs, dtype=float)
    tmp["yhat"] = (tmp["p"] >= float(thr)).astype(int)

    rows = []
    for cid, g in tmp.groupby("capture_id"):
        pos = g[g["label"] == 1]
        if len(pos) == 0:
            continue
        tp = int((pos["yhat"] == 1).sum())
        fn = int((pos["yhat"] == 0).sum())
        rec = tp / (tp + fn + 1e-9)
        rows.append((str(cid), rec, len(pos)))
    rows.sort(key=lambda x: x[1])
    return rows[:10]

worst_iscx = per_capture_recall(iscx_test_df, p_iscx, thr)
print("Worst 10 ISCX captures (capture_id, recall, #pos_flows):")
for cid, rec, npos in worst_iscx:
    print(cid, rec, npos)

Worst 10 ISCX captures (capture_id, recall, #pos_flows):
vpn_vpn_voipbuster1a.pcap 0.0 58
vpn_vpn_ftps_a.pcap 0.6517857142798947 112
vpn_vpn_hangouts_chat1b.pcap 0.9032258064370448 62
vpn_vpn_aim_chat1b.pcap 0.9999999998333333 6


In [6]:
# DIAGNOSE VOIPBUSTER
vb_mask = iscx_test_df["capture_id"].str.contains("voipbuster")
vb_df = iscx_test_df[vb_mask].copy()

X_vb = X_all.loc[vb_df.index, feat_cols].to_numpy(dtype=float)
p_vb = booster.predict(xgb.DMatrix(X_vb, feature_names=feat_cols))

print(f"\nVoipbuster flows: {len(vb_df)}")
print(f"Mean Prob: {p_vb.mean():.4f} (Threshold: {thr:.4f})")
print(f"Max Prob:  {p_vb.max():.4f}")

print("\n--- Feature Comparison: Voipbuster vs All VPN ---")
vpn_mean = df_all[df_all["label"]==1][feat_cols].mean()
vb_mean = vb_df[feat_cols].mean()

diffs = []
for c in feat_cols:
    if abs(vpn_mean[c]) > 1e-9:
        ratio = vb_mean[c] / (vpn_mean[c] + 1e-9)
        if ratio < 0.5 or ratio > 2.0:
            diffs.append((c, ratio, vb_mean[c], vpn_mean[c]))

diffs.sort(key=lambda x: abs(np.log(x[1] + 1e-9)), reverse=True)

print(f"{'Feature':<30} | {'Ratio':<6} | {'Voipbuster':<10} | {'Avg VPN':<10}")
for c, r, v, a in diffs[:15]:
    print(f"{c:<30} | {r:.2f}   | {v:10.4f} | {a:10.4f}")



Voipbuster flows: 130
Mean Prob: 0.1062 (Threshold: 0.9528)
Max Prob:  0.9268

--- Feature Comparison: Voipbuster vs All VPN ---
Feature                        | Ratio  | Voipbuster | Avg VPN   
f_pkt_imbalance                | -0.42   |     0.1992 |    -0.4728
f_byte_imbalance               | -0.28   |     0.1575 |    -0.5604
f_iat_burstiness               | -2.92   |    -0.3475 |     0.1189
sz_all_std                     | -0.68   |    -0.0942 |     0.1393
sz_all_p25                     | -5.00   |     0.6470 |    -0.1295
sz_mean_min                    | 136.07   |     0.6476 |     0.0048
sz_std_max                     | -4.87   |    -0.1254 |     0.0257
sz_std_min                     | 0.04   |     0.0013 |     0.0317
h_size_all_00                  | -3.32   |    -0.8481 |     0.2557
h_size_all_02                  | -2.89   |     0.7392 |    -0.2561
h_size_all_04                  | -0.50   |    -0.0661 |     0.1315
h_size_all_05                  | -0.35   |    -0.1282 |     0.3615


C:\Users\scoti\AppData\Local\Temp\ipykernel_18828\2720048575.py:23: RuntimeWarning: invalid value encountered in log
  diffs.sort(key=lambda x: abs(np.log(x[1] + 1e-9)), reverse=True)


In [7]:
# CHECK TRAINING SET FOR VOIPBUSTER
train_df = df_all[df_all["split"] == "train"]
vb_train = train_df[train_df["capture_id"].str.contains("voipbuster")]
print(f"\nVoipbuster flows in TRAIN: {len(vb_train)}")
if len(vb_train) > 0:
    print("It IS in training. The model is just failing to learn it (likely due to feature scaling/outliers).")
else:
    print("It is NOT in training. This is a generalization failure.")



Voipbuster flows in TRAIN: 280
It IS in training. The model is just failing to learn it (likely due to feature scaling/outliers).
